## 1. Imports and File Directories

In [1]:
import os
import re
import gc
 
import h5py
import numpy as np
 
from sklearn.model_selection import train_test_split
 
DATA_DIR = '../data/'
 
EXP_DIR_PATTERN = re.compile(r'^exp-[A-C]-[1-3]$')
 
exp_folders = sorted(
    d for d in os.listdir(DATA_DIR)
    if os.path.isdir(os.path.join(DATA_DIR, d)) and EXP_DIR_PATTERN.match(d)
)
 
print(f"Found {len(exp_folders)} exp-k-i folder(s):")
for d in exp_folders:
    print(f"  {d}")

Found 6 exp-k-i folder(s):
  exp-A-1
  exp-A-2
  exp-A-3
  exp-B-1
  exp-B-2
  exp-B-3


In [2]:
def clean_path(exp_key, scenario):
    # Must match the paths written by the cleaning notebook
    return os.path.join(DATA_DIR, exp_key, 'processed', f'{scenario}_clean.h5')
 
 
def splits_dir(exp_key):
    d = os.path.join(DATA_DIR, exp_key, 'splits')
    os.makedirs(d, exist_ok=True)
    return d
 
 
def splits_3dom_dir(exp_key):
    d = os.path.join(splits_dir(exp_key), 'rma_umi_uma')
    os.makedirs(d, exist_ok=True)
    return d
 
 
def save_split(path, data, mods, snrs, domain, idx):
    with h5py.File(path, 'w') as f:
        f.create_dataset('Data',   data=data[idx])
        f.create_dataset('Mods',   data=mods[idx])
        f.create_dataset('SNRs',   data=snrs[idx])
        f.create_dataset('Domain', data=domain[idx])

## 2. Process each exp-k-i: 2-domain split (Rma + Umi), then 3-domain split (Rma+Umi+Uma)

In [3]:
for exp_key in exp_folders:
    print(f"\n########## Splitting {exp_key} ##########")
 
    # ---------------------------------------------------------------
    # 2a. Two-domain split: Rma + Umi
    # ---------------------------------------------------------------
    with h5py.File(clean_path(exp_key, 'Rma'), 'r') as f:
        rma_data, rma_mods, rma_snrs = f['Data'][:], f['Mods'][:], f['SNRs'][:]
    rma_domain = np.zeros(len(rma_snrs), dtype=np.int8)  # 0 = Rma
 
    with h5py.File(clean_path(exp_key, 'Umi'), 'r') as f:
        umi_data, umi_mods, umi_snrs = f['Data'][:], f['Mods'][:], f['SNRs'][:]
    umi_domain = np.ones(len(umi_snrs), dtype=np.int8)  # 1 = Umi
 
    all_data   = np.concatenate([rma_data, umi_data])
    all_mods   = np.concatenate([rma_mods, umi_mods])
    all_snrs   = np.concatenate([rma_snrs, umi_snrs])
    all_domain = np.concatenate([rma_domain, umi_domain])
 
    print(f"{exp_key} — combined 2-domain dataset: {all_data.shape[0]} rows")
 
    del rma_data, rma_mods, rma_snrs, umi_data, umi_mods, umi_snrs
    gc.collect()
 
    # -- Stratify --
    mod_idx = np.argmax(all_mods, axis=1)
    all_idx = np.arange(len(mod_idx))
 
    strat_labels = np.array([
        f"{m}_{s}_{d}" for m, s, d in zip(mod_idx, all_snrs, all_domain)
    ])
 
    train_idx, temp_idx = train_test_split(
        all_idx, train_size=0.8, stratify=strat_labels, random_state=42
    )
    val_idx, test_idx = train_test_split(
        temp_idx, train_size=0.5, stratify=strat_labels[temp_idx], random_state=42
    )
 
    print(f"{exp_key} 2-domain — Train: {len(train_idx)}  Val: {len(val_idx)}  Test: {len(test_idx)}")
 
    # -- Sanity check: confirm domain ratio is preserved in each split --
    for name, idx in [('train', train_idx), ('val', val_idx), ('test', test_idx)]:
        domain_ratio = all_domain[idx].mean()
        mod_counts = np.bincount(mod_idx[idx], minlength=5) / len(idx)
        snr_vals, snr_counts = np.unique(all_snrs[idx], return_counts=True)
        snr_frac_range = np.ptp(snr_counts / snr_counts.sum())
        print(f"  {exp_key}/{name}: domain frac Umi={domain_ratio:.3f} (~0.5) | "
              f"mod frac range={mod_counts.min():.3f}-{mod_counts.max():.3f} (~0.2 each) | "
              f"snr frac spread={snr_frac_range:.4f} (should be ~0)")
 
    # -- Save --
    out_dir = splits_dir(exp_key)
    save_split(os.path.join(out_dir, 'train.h5'), all_data, all_mods, all_snrs, all_domain, train_idx)
    save_split(os.path.join(out_dir, 'val.h5'),   all_data, all_mods, all_snrs, all_domain, val_idx)
    save_split(os.path.join(out_dir, 'test.h5'),  all_data, all_mods, all_snrs, all_domain, test_idx)
    print(f"  saved train.h5, val.h5, test.h5 -> {out_dir}")
 
    del all_data, all_mods, all_snrs, all_domain, mod_idx, all_idx, strat_labels
    del train_idx, val_idx, test_idx, temp_idx
    gc.collect()
 
    # ---------------------------------------------------------------
    # 2b. Three-domain split: Rma + Umi + Uma
    # ---------------------------------------------------------------
    with h5py.File(clean_path(exp_key, 'Rma'), 'r') as f:
        rma_data, rma_mods, rma_snrs = f['Data'][:], f['Mods'][:], f['SNRs'][:]
    rma_domain_3 = np.zeros(len(rma_snrs), dtype=np.int8)  # 0 = Rma
 
    with h5py.File(clean_path(exp_key, 'Umi'), 'r') as f:
        umi_data, umi_mods, umi_snrs = f['Data'][:], f['Mods'][:], f['SNRs'][:]
    umi_domain_3 = np.ones(len(umi_snrs), dtype=np.int8)  # 1 = Umi
 
    with h5py.File(clean_path(exp_key, 'Uma'), 'r') as f:
        uma_data, uma_mods, uma_snrs = f['Data'][:], f['Mods'][:], f['SNRs'][:]
    uma_domain_3 = np.full(len(uma_snrs), 2, dtype=np.int8)  # 2 = Uma
 
    all_data_full   = np.concatenate([rma_data, umi_data, uma_data])
    all_mods_full   = np.concatenate([rma_mods, umi_mods, uma_mods])
    all_snrs_full   = np.concatenate([rma_snrs, umi_snrs, uma_snrs])
    all_domain_full = np.concatenate([rma_domain_3, umi_domain_3, uma_domain_3])
 
    print(f"{exp_key} — combined 3-domain dataset: {all_data_full.shape[0]} rows")
 
    del rma_data, rma_mods, rma_snrs, umi_data, umi_mods, umi_snrs, uma_data, uma_mods, uma_snrs
    gc.collect()
 
    # -- Stratify --
    mod_idx_full = np.argmax(all_mods_full, axis=1)
    all_idx_full = np.arange(len(mod_idx_full))
 
    strat_labels_full = np.array([
        f"{m}_{s}_{d}" for m, s, d in zip(mod_idx_full, all_snrs_full, all_domain_full)
    ])
 
    train_idx_full, temp_idx_full = train_test_split(
        all_idx_full, train_size=0.8, stratify=strat_labels_full, random_state=42
    )
    val_idx_full, test_idx_full = train_test_split(
        temp_idx_full, train_size=0.5,
        stratify=strat_labels_full[temp_idx_full], random_state=42
    )
 
    print(f"{exp_key} 3-domain — Train: {len(train_idx_full)}  Val: {len(val_idx_full)}  Test: {len(test_idx_full)}")
 
    # -- Sanity check --
    for name, idx in [('train', train_idx_full), ('val', val_idx_full), ('test', test_idx_full)]:
        vals, counts = np.unique(all_domain_full[idx], return_counts=True)
        fracs = counts / counts.sum()
        breakdown = ", ".join(f"domain {v}={f:.3f}" for v, f in zip(vals, fracs))
        print(f"  {exp_key}/{name}: {breakdown}  (should be ~0.333 each)")
 
    # -- Save --
    out_dir_3dom = splits_3dom_dir(exp_key)
    save_split(os.path.join(out_dir_3dom, 'train.h5'), all_data_full, all_mods_full, all_snrs_full, all_domain_full, train_idx_full)
    save_split(os.path.join(out_dir_3dom, 'val.h5'),   all_data_full, all_mods_full, all_snrs_full, all_domain_full, val_idx_full)
    save_split(os.path.join(out_dir_3dom, 'test.h5'),  all_data_full, all_mods_full, all_snrs_full, all_domain_full, test_idx_full)
    print(f"  saved train.h5, val.h5, test.h5 -> {out_dir_3dom}")
 
    del all_data_full, all_mods_full, all_snrs_full, all_domain_full
    del mod_idx_full, all_idx_full, strat_labels_full
    del train_idx_full, val_idx_full, test_idx_full, temp_idx_full
    gc.collect()
 
    print(f"########## Done splitting {exp_key} ##########")
 


########## Splitting exp-A-1 ##########
exp-A-1 — combined 2-domain dataset: 163840 rows
exp-A-1 2-domain — Train: 131072  Val: 16384  Test: 16384
  exp-A-1/train: domain frac Umi=0.500 (~0.5) | mod frac range=0.200-0.200 (~0.2 each) | snr frac spread=0.0000 (should be ~0)
  exp-A-1/val: domain frac Umi=0.500 (~0.5) | mod frac range=0.200-0.200 (~0.2 each) | snr frac spread=0.0002 (should be ~0)
  exp-A-1/test: domain frac Umi=0.500 (~0.5) | mod frac range=0.200-0.200 (~0.2 each) | snr frac spread=0.0003 (should be ~0)
  saved train.h5, val.h5, test.h5 -> ../data/exp-A-1/splits
exp-A-1 — combined 3-domain dataset: 245760 rows
exp-A-1 3-domain — Train: 196608  Val: 24576  Test: 24576
  exp-A-1/train: domain 0=0.333, domain 1=0.333, domain 2=0.333  (should be ~0.333 each)
  exp-A-1/val: domain 0=0.333, domain 1=0.333, domain 2=0.333  (should be ~0.333 each)
  exp-A-1/test: domain 0=0.333, domain 1=0.333, domain 2=0.333  (should be ~0.333 each)
  saved train.h5, val.h5, test.h5 -> ../dat